# Track-A prospective paired zero-shot audit v1

Diagnostic-only selected-slice development evidence. This notebook does not train, fine-tune, or select a checkpoint. Outputs remain private; only `sanitized_handoff.json` is suitable for handoff.

In [ ]:
# Actionable GPU preflight must run before clone or install.
import json, os, platform, shutil, subprocess
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No nvidia-smi found. In Colab choose Runtime > Change runtime type > GPU, then reconnect.')
gpu_query = subprocess.run([
    'nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap',
    '--format=csv,noheader,nounits'
], check=True, capture_output=True, text=True).stdout.strip()
if not gpu_query:
    raise RuntimeError('nvidia-smi returned no GPU. Reconnect to a GPU runtime before continuing.')
print(json.dumps({'runtime': 'Google Colab GPU', 'python': platform.python_version(), 'gpu': gpu_query}, indent=2))


In [ ]:
# Safe to rerun in the same kernel: reuse the clone and detach at the exact audit commit.
from pathlib import Path
REPOSITORY_URL = input('Repository clone URL: ').strip()
EXPECTED_AUDIT_COMMIT = input('Exact pushed audit code commit SHA: ').strip().lower()
if len(EXPECTED_AUDIT_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in EXPECTED_AUDIT_COMMIT):
    raise ValueError('Expected a full 40-character commit SHA.')
REPO_DIR = Path('/content/MRIxFields')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPOSITORY_URL], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone; choose a fresh runtime or move it manually.')
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'cat-file', '-e', EXPECTED_AUDIT_COMMIT + '^{commit}'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', EXPECTED_AUDIT_COMMIT], check=True)
actual = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
if actual != EXPECTED_AUDIT_COMMIT:
    raise RuntimeError(f'Checkout mismatch: {actual} != {EXPECTED_AUDIT_COMMIT}')
print('Verified audit checkout:', actual)


In [ ]:
# Editable install and same-kernel import repair.
import importlib, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR) + '[dev,nifti]'], check=True)
SOURCE_DIR = str(REPO_DIR / 'src')
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)
sys.path.insert(0, SOURCE_DIR)
for module_name in list(sys.modules):
    if module_name == 'fieldbridge' or module_name.startswith('fieldbridge.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge as installed_fieldbridge
PACKAGE_FILE = Path(installed_fieldbridge.__file__).resolve()
if REPO_DIR not in PACKAGE_FILE.parents:
    raise RuntimeError(f'FieldBridge imported from the wrong path: {PACKAGE_FILE}')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access CUDA. Reconnect to a GPU runtime before running the audit.')
print({'fieldbridge_package': str(PACKAGE_FILE), 'torch': torch.__version__, 'cuda': torch.version.cuda})


In [ ]:
# Private inputs and output location; do not copy these values into Git.
from google.colab import drive
drive.mount('/content/drive')
MANIFEST_PATH = Path(input('Private official JSONL manifest path: ').strip())
CHECKPOINT_PATH = Path(input('Frozen residual checkpoint path: ').strip())
OUTPUT_DIR = Path(input('Private output directory: ').strip())
for required in (MANIFEST_PATH, CHECKPOINT_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Frozen diagnostic: inference only.
command = [
    sys.executable, str(REPO_DIR / 'notebooks/prospective_paired_zero_shot_runner.py'),
    '--manifest', str(MANIFEST_PATH),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--config', str(REPO_DIR / 'configs/experiment/prospective_paired_zero_shot_v1.yaml'),
    '--device', 'cuda',
]
subprocess.run(command, cwd=REPO_DIR, check=True)
handoff = json.loads((OUTPUT_DIR / 'sanitized_handoff.json').read_text())
print(json.dumps(handoff, indent=2, sort_keys=True))
